# 0. Overview

This notebook analyzes the 850 hPa streamfunction and rotational wind ENSO composites over the Maritime Continent and Indo-Pacific, split into **Full (1981-2025)**, **Past (1981-2006)**, and **Recent (2007-2025)** periods.

Scientific objective:
- Separate the baseline streamfunction state from ENSO-conditioned responses.
- Plot streamfunction anomaly as the shaded field and rotational wind anomaly as vector overlays.


# 1. Import Library


In [ ]:
import re
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import dask
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter


In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'


# 2. Open Data & Pre-Process


#### Open processed 850 hPa streamfunction / rotational wind


In [ ]:
psi_candidates = [
    '../../external/ClimateData/era5-monthly/psi_chi_850_global_1979-2025.nc',
]

def _normalize_name(name):
    return re.sub(r'[^a-z0-9]+', '', str(name).lower())

def _first_existing(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f'None of these files exist: {candidates}')

def _pick_data_var(ds, exact_candidates, required_tokens=()):
    exact_map = {_normalize_name(name): name for name in ds.data_vars}
    for candidate in exact_candidates:
        key = _normalize_name(candidate)
        if key in exact_map:
            return ds[exact_map[key]]
    for name in ds.data_vars:
        norm = _normalize_name(name)
        if all(token in norm for token in required_tokens):
            return ds[name]
    raise KeyError(f'Could not find a data variable for {exact_candidates}')

def _pick_numeric_column(df, exclude=()):
    exclude_norm = {_normalize_name(name) for name in exclude}
    numeric_candidates = [
        col for col in df.columns
        if _normalize_name(col) not in exclude_norm and pd.api.types.is_numeric_dtype(df[col])
    ]
    for key in ('nino34', 'nino', 'anom'):
        for col in numeric_candidates:
            if key in _normalize_name(col):
                return col
    if numeric_candidates:
        return numeric_candidates[0]
    raise KeyError('Could not find a numeric Niño-3.4 column')

def _nice_symmetric_limit(value):
    value = float(value)
    if not np.isfinite(value) or value <= 0:
        return 1.0
    exponent = np.floor(np.log10(value))
    step = 5 * (10 ** (exponent - 1))
    return np.ceil(value / step) * step

ds_psi = xr.open_dataset(_first_existing(psi_candidates), chunks={'time': 12})
rename_map = {}
if 'valid_time' in ds_psi.dims or 'valid_time' in ds_psi.coords:
    rename_map['valid_time'] = 'time'
if 'latitude' in ds_psi.dims or 'latitude' in ds_psi.coords:
    rename_map['latitude'] = 'lat'
if 'longitude' in ds_psi.dims or 'longitude' in ds_psi.coords:
    rename_map['longitude'] = 'lon'
if rename_map:
    ds_psi = ds_psi.rename(rename_map)

for level_name in ('pressure_level', 'level', 'isobaricInhPa'):
    if level_name in ds_psi.coords or level_name in ds_psi.dims:
        level_coord = ds_psi[level_name]
        if level_coord.ndim == 0:
            if float(level_coord) != 850:
                raise ValueError(f'Expected {level_name}=850, got {float(level_coord)}')
        else:
            ds_psi = ds_psi.sel({level_name: 850})
        break

psi = _pick_data_var(ds_psi, ['psi', 'streamfunction', 'stream_function'], required_tokens=('psi',)).rename('psi')
u_psi = _pick_data_var(ds_psi, ['u_psi', 'upsi', 'u_streamfunction', 'u_rotational'], required_tokens=('u', 'psi')).rename('u_psi')
v_psi = _pick_data_var(ds_psi, ['v_psi', 'vpsi', 'v_streamfunction', 'v_rotational'], required_tokens=('v', 'psi')).rename('v_psi')
psi_units = psi.attrs.get('units', 'm2 s-1')

if float(psi.lon.max()) <= 180:
    psi = psi.assign_coords(lon=(psi.lon % 360)).sortby('lon')
    u_psi = u_psi.assign_coords(lon=(u_psi.lon % 360)).sortby('lon')
    v_psi = v_psi.assign_coords(lon=(v_psi.lon % 360)).sortby('lon')

lat0 = float(psi.lat.isel(lat=0))
lat1 = float(psi.lat.isel(lat=-1))
lat_slice = slice(31, -31) if lat0 > lat1 else slice(-31, 31)

psi = psi.sel(time=slice('1980-12-01', '2025-02-01'), lat=lat_slice, lon=slice(60, 291))
u_psi = u_psi.sel(time=slice('1980-12-01', '2025-02-01'), lat=lat_slice, lon=slice(60, 291))
v_psi = v_psi.sel(time=slice('1980-12-01', '2025-02-01'), lat=lat_slice, lon=slice(60, 291))

month_mask = psi.time.dt.month.isin([12, 1, 2])
djf_year = xr.where(psi.time.dt.month == 12, psi.time.dt.year + 1, psi.time.dt.year)
psi = psi.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))
u_psi = u_psi.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))
v_psi = v_psi.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))

print('Using streamfunction file:', _first_existing(psi_candidates))
print('Selected variables:', psi.name, u_psi.name, v_psi.name)
print('Streamfunction units:', psi_units)


#### Open NINO34 index


In [ ]:
nino_candidates = [
    '../../external/ClimateData/index-monthly/nino34.anom.csv',
]

nino_path = _first_existing(nino_candidates)
df_nino34 = pd.read_csv(nino_path)
df_nino34.columns = [str(col).strip() for col in df_nino34.columns]

if 'Date' in df_nino34.columns or 'date' in df_nino34.columns:
    date_col = 'Date' if 'Date' in df_nino34.columns else 'date'
    value_col = _pick_numeric_column(df_nino34, exclude=(date_col,))
    df_nino34[date_col] = pd.to_datetime(df_nino34[date_col])
    df_nino34 = df_nino34.set_index(date_col).loc['1980-12-01':'2025-03-01']
    df_nino34 = df_nino34[df_nino34.index.month.isin([12, 1, 2])].copy()
    df_nino34['djf_year'] = df_nino34.index.year + (df_nino34.index.month == 12).astype('int16')
    df_nino34 = df_nino34[['djf_year', value_col]].rename(columns={value_col: 'nino34'})
elif 'djf_year' in df_nino34.columns:
    value_col = _pick_numeric_column(df_nino34, exclude=('djf_year',))
    df_nino34 = df_nino34[['djf_year', value_col]].rename(columns={value_col: 'nino34'})
elif 'year' in df_nino34.columns:
    value_col = _pick_numeric_column(df_nino34, exclude=('year',))
    df_nino34 = df_nino34[['year', value_col]].rename(columns={'year': 'djf_year', value_col: 'nino34'})
else:
    value_col = _pick_numeric_column(df_nino34)
    df_nino34 = df_nino34.iloc[:, :2].copy()
    df_nino34.columns = ['djf_year', 'nino34'] if len(df_nino34.columns) == 2 else ['djf_year', 'nino34']

df_nino34['DJF'] = 'DJF' + df_nino34['djf_year'].astype(str)
print('Using Niño-3.4 file:', nino_path)
print('Niño-3.4 columns normalized to:', df_nino34.columns.tolist())


#### Define Period


In [ ]:
full_years = np.arange(1981, 2026)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2026)

psi_clim = psi.sel(time=psi.djf_year.isin(full_years))
psi_past = psi_clim.sel(time=psi_clim.djf_year.isin(past_years))
psi_recent = psi_clim.sel(time=psi_clim.djf_year.isin(recent_years))

u_psi_clim = u_psi.sel(time=u_psi.djf_year.isin(full_years))
u_psi_past = u_psi_clim.sel(time=u_psi_clim.djf_year.isin(past_years))
u_psi_recent = u_psi_clim.sel(time=u_psi_clim.djf_year.isin(recent_years))

v_psi_clim = v_psi.sel(time=v_psi.djf_year.isin(full_years))
v_psi_past = v_psi_clim.sel(time=v_psi_clim.djf_year.isin(past_years))
v_psi_recent = v_psi_clim.sel(time=v_psi_clim.djf_year.isin(recent_years))

nino34_clim = df_nino34[df_nino34['djf_year'].isin(full_years)].copy()
nino34_past = nino34_clim[nino34_clim['djf_year'].isin(past_years)].copy()
nino34_recent = nino34_clim[nino34_clim['djf_year'].isin(recent_years)].copy()

period_coord = xr.DataArray(
    np.where(psi_clim.djf_year <= 2006, 'past', 'recent'),
    coords={'time': psi_clim.time},
    dims='time',
    name='period',
)
psi_period = psi_clim.assign_coords(period=period_coord)
u_psi_period = u_psi_clim.assign_coords(period=period_coord)
v_psi_period = v_psi_clim.assign_coords(period=period_coord)


#### Define Area


In [ ]:
# Define the extent and ticks 
# ip = Indonesia-Pacific region
ip_extent = [60, 270, -30, 30]
ip_yticks = np.arange(-30, 31, 10)
ip_xticks = np.arange(60, 271, 30)


In [ ]:
# Define the extent and ticks 
# mc: maritime continent
mc_extent = [80, 160, -20, 20]  # [lon_min, lon_max, lat_min, lat_max]
mc_yticks = np.arange(-20, 21, 10)
mc_xticks = np.arange(90, 161, 20)


In [ ]:
# Compute the shared climatology means once and reuse the period grouping
full_psi_plot, period_means, full_u_psi_plot, period_u_means, full_v_psi_plot, period_v_means = dask.compute(
    psi_clim.mean('time'),
    psi_period.groupby('period').mean('time'),
    u_psi_clim.mean('time'),
    u_psi_period.groupby('period').mean('time'),
    v_psi_clim.mean('time'),
    v_psi_period.groupby('period').mean('time'),
)

# Transpose after loading (faster on in-memory numpy arrays)
full_psi_plot = full_psi_plot.transpose('lat', 'lon')
past_psi_plot = period_means.sel(period='past').transpose('lat', 'lon')
recent_psi_plot = period_means.sel(period='recent').transpose('lat', 'lon')
diff_psi_plot = recent_psi_plot - past_psi_plot

full_u_psi_plot = full_u_psi_plot.transpose('lat', 'lon')
past_u_psi_plot = period_u_means.sel(period='past').transpose('lat', 'lon')
recent_u_psi_plot = period_u_means.sel(period='recent').transpose('lat', 'lon')
diff_u_psi_plot = recent_u_psi_plot - past_u_psi_plot

full_v_psi_plot = full_v_psi_plot.transpose('lat', 'lon')
past_v_psi_plot = period_v_means.sel(period='past').transpose('lat', 'lon')
recent_v_psi_plot = period_v_means.sel(period='recent').transpose('lat', 'lon')
diff_v_psi_plot = recent_v_psi_plot - past_v_psi_plot


In [ ]:
def _add_quiver(ax, u, v, step=16, scale=70, width=0.002, color='k', key_u=10, key_y=1.1, key_label=None, key_color='k'):
    u_plot = u.isel(lat=slice(None, None, step), lon=slice(None, None, step))
    v_plot = v.isel(lat=slice(None, None, step), lon=slice(None, None, step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=width,
        color=color,
        zorder=4,
    )
    if key_label is None:
        key_label = f'{key_u} m/s'
    ax.quiverkey(q, X=0.88, Y=key_y, U=key_u, label=key_label, labelpos='E', coordinates='axes', color=key_color)
    return q


def add_quiver_clim(ax, u, v, step=16, scale=70, width=0.002, color='k', key_y=1.1, key_u=10, key_label=None):
    return _add_quiver(ax, u, v, step=step, scale=scale, width=width, color=color, key_u=key_u, key_y=key_y, key_label=key_label)


# 4. Plot Composite


### A. Analisis El Nino


In [ ]:
# define el nino years based on DJF-mean nino34 index threshold
elnino_threshold = 0.5
nino34_clim_djf = nino34_clim.groupby('djf_year')['nino34'].mean().reset_index()
nino34_past_djf = nino34_past.groupby('djf_year')['nino34'].mean().reset_index()
nino34_recent_djf = nino34_recent.groupby('djf_year')['nino34'].mean().reset_index()

nino34_clim_elnino = nino34_clim_djf[nino34_clim_djf['nino34'] >= elnino_threshold]
nino34_past_elnino = nino34_past_djf[nino34_past_djf['nino34'] >= elnino_threshold]
nino34_recent_elnino = nino34_recent_djf[nino34_recent_djf['nino34'] >= elnino_threshold]

elnino_years_clim = sorted(nino34_clim_elnino['djf_year'].tolist())
elnino_years_past = sorted(nino34_past_elnino['djf_year'].tolist())
elnino_years_recent = sorted(nino34_recent_elnino['djf_year'].tolist())
print('El Nino DJF years (1981-2025):', elnino_years_clim)
print('El Nino DJF years (1981-2006):', elnino_years_past)
print('El Nino DJF years (2007-2025):', elnino_years_recent)

psi_clim_elnino = psi_clim.sel(time=psi_clim.djf_year.isin(elnino_years_clim)).mean('time').transpose('lat', 'lon')
psi_past_elnino = psi_past.sel(time=psi_past.djf_year.isin(elnino_years_past)).mean('time').transpose('lat', 'lon')
psi_recent_elnino = psi_recent.sel(time=psi_recent.djf_year.isin(elnino_years_recent)).mean('time').transpose('lat', 'lon')

u_psi_clim_elnino = u_psi_clim.sel(time=u_psi_clim.djf_year.isin(elnino_years_clim)).mean('time').transpose('lat', 'lon')
u_psi_past_elnino = u_psi_past.sel(time=u_psi_past.djf_year.isin(elnino_years_past)).mean('time').transpose('lat', 'lon')
u_psi_recent_elnino = u_psi_recent.sel(time=u_psi_recent.djf_year.isin(elnino_years_recent)).mean('time').transpose('lat', 'lon')

v_psi_clim_elnino = v_psi_clim.sel(time=v_psi_clim.djf_year.isin(elnino_years_clim)).mean('time').transpose('lat', 'lon')
v_psi_past_elnino = v_psi_past.sel(time=v_psi_past.djf_year.isin(elnino_years_past)).mean('time').transpose('lat', 'lon')
v_psi_recent_elnino = v_psi_recent.sel(time=v_psi_recent.djf_year.isin(elnino_years_recent)).mean('time').transpose('lat', 'lon')

psi_clim_elnino_anom = (psi_clim_elnino - full_psi_plot).reset_coords(drop=True)
psi_past_elnino_anom = (psi_past_elnino - past_psi_plot).reset_coords(drop=True)
psi_recent_elnino_anom = (psi_recent_elnino - recent_psi_plot).reset_coords(drop=True)
diff_psi_elnino_anom = (psi_recent_elnino_anom - psi_past_elnino_anom).reset_coords(drop=True)

u_psi_clim_elnino_anom = (u_psi_clim_elnino - full_u_psi_plot).reset_coords(drop=True)
u_psi_past_elnino_anom = (u_psi_past_elnino - past_u_psi_plot).reset_coords(drop=True)
u_psi_recent_elnino_anom = (u_psi_recent_elnino - recent_u_psi_plot).reset_coords(drop=True)
diff_u_psi_elnino_anom = (u_psi_recent_elnino_anom - u_psi_past_elnino_anom).reset_coords(drop=True)

v_psi_clim_elnino_anom = (v_psi_clim_elnino - full_v_psi_plot).reset_coords(drop=True)
v_psi_past_elnino_anom = (v_psi_past_elnino - past_v_psi_plot).reset_coords(drop=True)
v_psi_recent_elnino_anom = (v_psi_recent_elnino - recent_v_psi_plot).reset_coords(drop=True)
diff_v_psi_elnino_anom = (v_psi_recent_elnino_anom - v_psi_past_elnino_anom).reset_coords(drop=True)

elnino_limit = _nice_symmetric_limit(
    xr.concat(
        [
            np.abs(psi_clim_elnino_anom),
            np.abs(psi_past_elnino_anom),
            np.abs(psi_recent_elnino_anom),
            np.abs(diff_psi_elnino_anom),
        ],
        dim='stack',
    ).max().compute().item()
)
elnino_levels = np.linspace(-elnino_limit, elnino_limit, 13)
elnino_ticks = np.linspace(-elnino_limit, elnino_limit, 7)


#### plot komposit indo pasifik


In [ ]:
# data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (psi_clim_elnino_anom, u_psi_clim_elnino_anom, v_psi_clim_elnino_anom, 'Komposit Anomali Streamfunction 850 hPa  El Nino DJF 1981-2025', 'Spectral', elnino_levels, elnino_ticks, f'Anomali streamfunction 850 hPa ({psi_units})', 'both'),
    (psi_past_elnino_anom, u_psi_past_elnino_anom, v_psi_past_elnino_anom, 'Komposit Anomali Streamfunction 850 hPa El Nino DJF 1981-2006', 'Spectral', elnino_levels, elnino_ticks, f'Anomali streamfunction 850 hPa ({psi_units})', 'both'),
    (psi_recent_elnino_anom, u_psi_recent_elnino_anom, v_psi_recent_elnino_anom, 'Komposit Anomali Streamfunction 850 hPa El Nino DJF 2007-2025', 'Spectral', elnino_levels, elnino_ticks, f'Anomali streamfunction 850 hPa ({psi_units})', 'both'),
    (diff_psi_elnino_anom, diff_u_psi_elnino_anom, diff_v_psi_elnino_anom, 'Selisih Komposit Anomali Streamfunction 850 hPa dan El Nino DJF P2 - P1', 'Spectral', elnino_levels, elnino_ticks, f'Selisih anomali streamfunction 850 hPa ({psi_units})', 'both'),
]

for data, u_data, v_data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_quiver_clim(ax, u_data, v_data, step=20, scale=50, width=0.0017, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


#### plot komposit maritime continent


In [ ]:
for data, u_data, v_data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_quiver_clim(ax, u_data, v_data, step=10, scale=50, width=0.002, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


### B. Analisis La Nina


In [ ]:
# define la nina years based on DJF-mean nino34 index threshold
lanina_threshold = -0.5

nino34_clim_lanina = nino34_clim_djf[nino34_clim_djf['nino34'] <= lanina_threshold]
nino34_past_lanina = nino34_past_djf[nino34_past_djf['nino34'] <= lanina_threshold]
nino34_recent_lanina = nino34_recent_djf[nino34_recent_djf['nino34'] <= lanina_threshold]

lanina_years_clim = sorted(nino34_clim_lanina['djf_year'].tolist())
lanina_years_past = sorted(nino34_past_lanina['djf_year'].tolist())
lanina_years_recent = sorted(nino34_recent_lanina['djf_year'].tolist())
print('La Nina DJF years (1981-2025):', lanina_years_clim)
print('La Nina DJF years (1981-2006):', lanina_years_past)
print('La Nina DJF years (2007-2025):', lanina_years_recent)

psi_clim_lanina = psi_clim.sel(time=psi_clim.djf_year.isin(lanina_years_clim)).mean('time').transpose('lat', 'lon')
psi_past_lanina = psi_past.sel(time=psi_past.djf_year.isin(lanina_years_past)).mean('time').transpose('lat', 'lon')
psi_recent_lanina = psi_recent.sel(time=psi_recent.djf_year.isin(lanina_years_recent)).mean('time').transpose('lat', 'lon')

u_psi_clim_lanina = u_psi_clim.sel(time=u_psi_clim.djf_year.isin(lanina_years_clim)).mean('time').transpose('lat', 'lon')
u_psi_past_lanina = u_psi_past.sel(time=u_psi_past.djf_year.isin(lanina_years_past)).mean('time').transpose('lat', 'lon')
u_psi_recent_lanina = u_psi_recent.sel(time=u_psi_recent.djf_year.isin(lanina_years_recent)).mean('time').transpose('lat', 'lon')

v_psi_clim_lanina = v_psi_clim.sel(time=v_psi_clim.djf_year.isin(lanina_years_clim)).mean('time').transpose('lat', 'lon')
v_psi_past_lanina = v_psi_past.sel(time=v_psi_past.djf_year.isin(lanina_years_past)).mean('time').transpose('lat', 'lon')
v_psi_recent_lanina = v_psi_recent.sel(time=v_psi_recent.djf_year.isin(lanina_years_recent)).mean('time').transpose('lat', 'lon')

psi_clim_lanina_anom = (psi_clim_lanina - full_psi_plot).reset_coords(drop=True)
psi_past_lanina_anom = (psi_past_lanina - past_psi_plot).reset_coords(drop=True)
psi_recent_lanina_anom = (psi_recent_lanina - recent_psi_plot).reset_coords(drop=True)
diff_psi_lanina_anom = (psi_recent_lanina_anom - psi_past_lanina_anom).reset_coords(drop=True)

u_psi_clim_lanina_anom = (u_psi_clim_lanina - full_u_psi_plot).reset_coords(drop=True)
u_psi_past_lanina_anom = (u_psi_past_lanina - past_u_psi_plot).reset_coords(drop=True)
u_psi_recent_lanina_anom = (u_psi_recent_lanina - recent_u_psi_plot).reset_coords(drop=True)
diff_u_psi_lanina_anom = (u_psi_recent_lanina_anom - u_psi_past_lanina_anom).reset_coords(drop=True)

v_psi_clim_lanina_anom = (v_psi_clim_lanina - full_v_psi_plot).reset_coords(drop=True)
v_psi_past_lanina_anom = (v_psi_past_lanina - past_v_psi_plot).reset_coords(drop=True)
v_psi_recent_lanina_anom = (v_psi_recent_lanina - recent_v_psi_plot).reset_coords(drop=True)
diff_v_psi_lanina_anom = (v_psi_recent_lanina_anom - v_psi_past_lanina_anom).reset_coords(drop=True)

lanina_limit = _nice_symmetric_limit(
    xr.concat(
        [
            np.abs(psi_clim_lanina_anom),
            np.abs(psi_past_lanina_anom),
            np.abs(psi_recent_lanina_anom),
            np.abs(diff_psi_lanina_anom),
        ],
        dim='stack',
    ).max().compute().item()
)
lanina_levels = np.linspace(-lanina_limit, lanina_limit, 13)
lanina_ticks = np.linspace(-lanina_limit, lanina_limit, 7)


#### plot komposit indo pasifik


In [ ]:
# data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (psi_clim_lanina_anom, u_psi_clim_lanina_anom, v_psi_clim_lanina_anom, 'Komposit Anomali Streamfunction 850 hPa La Nina DJF 1981-2025', 'Spectral', lanina_levels, lanina_ticks, f'Anomali streamfunction 850 hPa ({psi_units})', 'both'),
    (psi_past_lanina_anom, u_psi_past_lanina_anom, v_psi_past_lanina_anom, 'Komposit Anomali Streamfunction 850 hPa La Nina DJF 1981-2006', 'Spectral', lanina_levels, lanina_ticks, f'Anomali streamfunction 850 hPa ({psi_units})', 'both'),
    (psi_recent_lanina_anom, u_psi_recent_lanina_anom, v_psi_recent_lanina_anom, 'Komposit Anomali Streamfunction 850 hPa La Nina DJF 2007-2025', 'Spectral', lanina_levels, lanina_ticks, f'Anomali streamfunction 850 hPa ({psi_units})', 'both'),
    (diff_psi_lanina_anom, diff_u_psi_lanina_anom, diff_v_psi_lanina_anom, 'Selisih Komposit Anomali Streamfunction 850 hPa La Nina DJF P2 - P1', 'Spectral', lanina_levels, lanina_ticks, f'Selisih anomali streamfunction 850 hPa ({psi_units})', 'both'),
]

for data, u_data, v_data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_quiver_clim(ax, u_data, v_data, step=20, scale=50, width=0.0017, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


#### plot komposit maritime continent


In [ ]:
for data, u_data, v_data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_quiver_clim(ax, u_data, v_data, step=10, scale=50, width=0.002, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()
